# День 09. Упражнение 04
# Пайплайны и ООП

## 0. Импорты

In [13]:
# Импорт библиотек и пояснения, где используются

import pandas as pd      # Работа с таблицами данных (используется на всех этапах)
import numpy as np       # Массивы и числовые операции (используется при вычислениях)
from sklearn.base import BaseEstimator, TransformerMixin  # Для создания своих трансформеров (FeatureExtractor, MyOneHotEncoder)
from sklearn.preprocessing import OneHotEncoder           # Кодирование категориальных признаков (MyOneHotEncoder)
from sklearn.model_selection import train_test_split, GridSearchCV  # Разделение данных (TrainValidationTest), подбор параметров (ModelSelection)
from sklearn.pipeline import Pipeline                     # Сборка пайплайна обработки (главная программа)
from tqdm.notebook import tqdm                            # Прогресс-бар при подборе параметров (ModelSelection)
import joblib                                             # Сохранение/загрузка модели

## 1. Пайплайн предобработки

Создайте три собственных трансформера, первые два из которых будут использоваться внутри [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. Класс `FeatureExtractor()`:
 - Принимает DataFrame с колонками `uid`, `labname`, `numTrials`, `timestamp` из файла [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Извлекает `hour` из `timestamp`.
 - Извлекает `weekday` из `timestamp` (число).
 - Удаляет колонку `timestamp`.
 - Возвращает новый DataFrame.

2. Класс `MyOneHotEncoder()`:
 - Принимает DataFrame после предыдущей трансформации и имя целевой колонки.
 - Находит все категориальные признаки и преобразует их с помощью `OneHotEncoder()`. Если целевая колонка тоже категориальная, её не преобразовывать.
 - Удаляет исходные категориальные признаки.
 - Возвращает DataFrame с признаками и Series с целевой колонкой.

3. Класс `TrainValidationTest()`:
 - Принимает `X` и `y`.
 - Возвращает `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, стратифицировано).


In [14]:
# 1. FeatureExtractor
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        df = X.copy()
        df['hour'] = pd.to_datetime(df['timestamp']).dt.hour
        df['dayofweek'] = pd.to_datetime(df['timestamp']).dt.dayofweek
        df = df.drop(columns=['timestamp'])
        return df

In [15]:
# 2. MyOneHotEncoder
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_col):
        self.target_col = target_col
        self.encoder = None
        self.cat_cols = None
    def fit(self, X, y=None):
        self.cat_cols = X.select_dtypes(include='object').columns.tolist()
        if self.target_col in self.cat_cols:
            self.cat_cols.remove(self.target_col)
        self.encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        self.encoder.fit(X[self.cat_cols])
        return self
    def transform(self, X):
        X_ = X.copy()
        y = X_[self.target_col]
        X_ = X_.drop(columns=[self.target_col])
        if self.cat_cols:
            encoded = self.encoder.transform(X_[self.cat_cols])
            encoded_df = pd.DataFrame(encoded, columns=self.encoder.get_feature_names_out(self.cat_cols), index=X_.index)
            X_ = X_.drop(columns=self.cat_cols)
            X_ = pd.concat([X_, encoded_df], axis=1)
        return X_, y

In [16]:
# 3. TrainValidationTest
class TrainValidationTest:
    def __init__(self, test_size=0.2, valid_size=0.25, random_state=21):
        self.test_size = test_size
        self.valid_size = valid_size
        self.random_state = random_state
    def split(self, X, y):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.test_size, random_state=self.random_state, stratify=y
        )
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train, y_train, test_size=self.valid_size, random_state=self.random_state, stratify=y_train
        )
        return X_train, X_valid, X_test, y_train, y_valid, y_test


## 2. Пайплайн выбора модели

Класс `ModelSelection()`

 - Принимает список экземпляров `GridSearchCV` и словарь, где ключи — индексы из этого списка, а значения — имена моделей, пример ниже (от высокого уровня к низкому):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), где jobs вы задаёте сами

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Метод `choose()` принимает `X_train`, `y_train`, `X_valid`, `y_valid` и возвращает имя лучшего классификатора по валидационной выборке.
 - Метод `best_results()` возвращает DataFrame с колонками `model`, `params`, `valid_score`, где строки — лучшие модели каждого класса.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - При переборе параметров класса модели выводите имя этого класса и прогресс через `tqdm.notebook`, в конце цикла выводите лучшую модель этого класса.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [17]:
# 4. ModelSelection
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.results = []
    def choose(self, X_train, y_train, X_valid, y_valid):
        best_score = -np.inf
        best_model = None
        best_name = None
        for idx, grid in enumerate(self.grids):
            name = self.grid_dict[idx]
            print(f"Estimator: {name}")
            grid.fit(X_train, y_train)
            train_score = grid.best_score_
            valid_score = grid.score(X_valid, y_valid)
            print(f"Best params: {grid.best_params_}")
            print(f"Best training accuracy: {train_score:.3f}")
            print(f"Validation set accuracy score for best params: {valid_score:.3f}\n")
            self.results.append({
                'model': name,
                'params': grid.best_params_,
                'valid_score': valid_score
            })
            if valid_score > best_score:
                best_score = valid_score
                best_model = grid.best_estimator_
                best_name = name
        print(f"Classifier with best validation set accuracy: {best_name}")
        self.best_model = best_model
        return best_model
    def best_results(self):
        return pd.DataFrame(self.results)


## 3. Финализация

Класс `Finalize()`
 - Принимает оценщик (estimator).
 - Метод `final_score()` принимает `X_train`, `y_train`, `X_test`, `y_test` и возвращает accuracy модели, пример:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Метод `save_model()` принимает путь, сохраняет модель по этому пути и выводит сообщение об успешном сохранении.

In [18]:
# 5. Finalize
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator
    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        acc = self.estimator.score(X_test, y_test)
        print(f"Accuracy of the final model is {acc}")
        return acc
    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print(f"Model saved to {path}")


## 4. Главная программа

1. Загрузите данные из файла (****имя файла****).
2. Создайте пайплайн предобработки из двух собственных трансформеров: `FeatureExtractor()` и `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Используйте этот пайплайн и его метод `fit_transform()` на исходном датасете.
```
data = preprocessing.fit_transform(df)
```
4. Получите `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` с помощью `TrainValidationTest()` и результата пайплайна.
5. Создайте экземпляр `ModelSelection()`, используйте метод `choose()` для нужных моделей и параметров, получите DataFrame лучших результатов.
6. Создайте экземпляр `Finalize()` с вашей лучшей моделью, используйте метод `final_score()` и сохраните модель в формате: `name_of_the_model_{accuracy on test dataset}.sav`.

Вот и всё, поздравляем!

In [20]:
# 1. Загрузка данных
df = pd.read_csv('../data/checker_submits.csv')  # укажи путь к своему файлу

In [21]:
# 2. Пайплайн предобработки
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('dayofweek'))  # было 'dayofweek', стало 'dayofweek'
])

# 3. Применение пайплайна
X, y = preprocessing.fit_transform(df)

# 4. Разделение на train/valid/test
splitter = TrainValidationTest()
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.split(X, y)


In [22]:
# 5. Подбор моделей
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

svm_params = [{
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None],
    'random_state': [21],
    'probability': [True]
}]
tree_params = [{
    'criterion': ['gini', 'entropy'],
    'max_depth': [5, 10, 15, 21],
    'class_weight': ['balanced', None],
    'random_state': [21]
}]
rf_params = [{
    'n_estimators': [10, 30, 50],
    'criterion': ['gini', 'entropy'],
    'max_depth': [10, 15, 22],
    'class_weight': ['balanced', None],
    'random_state': [21]
}]

gs_svm = GridSearchCV(SVC(), svm_params, scoring='accuracy', cv=2, n_jobs=-1)
gs_tree = GridSearchCV(DecisionTreeClassifier(), tree_params, scoring='accuracy', cv=2, n_jobs=-1)
gs_rf = GridSearchCV(RandomForestClassifier(), rf_params, scoring='accuracy', cv=2, n_jobs=-1)

grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}

selector = ModelSelection(grids, grid_dict)
best_model = selector.choose(X_train, y_train, X_valid, y_valid)
results_df = selector.best_results()
print(results_df)


Estimator: SVM
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.752
Validation set accuracy score for best params: 0.855

Estimator: Decision Tree
Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.802
Validation set accuracy score for best params: 0.864

Estimator: Random Forest
Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.854
Validation set accuracy score for best params: 0.875

Classifier with best validation set accuracy: Random Forest
           model                                             params  \
0            SVM  {'C': 10, 'class_weight': None, 'gamma': 'auto...   
1  Decision Tree  {'class_weight': None, 'criterion': 'gini', 'm...   
2  Random Forest  {'class_weight': None, 'criterion': 'gini', 'm...   

   valid_s

In [23]:
# 6. Финализация и сохранение модели
final = Finalize(best_model)
acc = final.final_score(X_train, y_train, X_test, y_test)
model_name = f"../data/{type(best_model).__name__}_{acc:.3f}.sav"
final.save_model(model_name)

Accuracy of the final model is 0.9023668639053254
Model saved to ../data/RandomForestClassifier_0.902.sav
